# RAY-IMAGE N2 — One-Click FIRST_IMAGE Experiment

This notebook runs the full controlled N2 experiment from start to finish: setup → smoke test → toy dataset → VAE training → N1 latent probe → 8,000-step whitened generator training → 12-image generation → evaluator → result summary.

**You only need to select a T4 GPU and use Runtime → Run all.**

The experiment uses the N1 statistics from the freshly trained VAE, so the run is self-contained and reproducible.

In [ ]:
# 1. Attach GPU and verify runtime
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU attached. In Colab choose Runtime → Change runtime type → T4 GPU, then Run all again.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# 2. Get the exact active project branch
%cd /content
!rm -rf anime-ai-companion
!git clone https://github.com/Rishidev-20thcenturey/anime-ai-companion.git
%cd /content/anime-ai-companion
!git fetch origin arena/01a07cdc-anime-ai-companion
!git checkout arena/01a07cdc-anime-ai-companion
!git rev-parse --short HEAD
!pip install -q -r requirements.txt
print('Active branch ready.')

In [ ]:
# 3. Project smoke test
!python -m ray_image.train_smoke
!python -m ray_image.probe_smoke
!python -m ray_image.whiten_smoke

In [ ]:
# 4. Fresh deterministic toy dataset
!rm -rf data/toy /content/n1 /content/ray_suite
!python tools/make_toy_dataset.py --output data/toy --samples 2048 --size 64 --seed 1337
print('Toy dataset ready.')

In [ ]:
# 5. Train the VAE (1200 steps)
!python -m ray_image.train_vae \
  --manifest data/toy/manifest.jsonl \
  --steps 1200 \
  --batch-size 32 \
  --save /content/ray_vae_v0_2.pt \
  --seed 0

In [ ]:
# 6. N1 latent probe — compute the exact stats used by N2
!mkdir -p /content/n1
!python -m ray_image.probe_vae_latents \
  --vae-checkpoint /content/ray_vae_v0_2.pt \
  --manifest data/toy/manifest.jsonl \
  --outdir /content/n1 \
  --size 64 \
  --seed 1337 \
  --stats-samples 512 \
  --stats-batch 32
print('N1 completed. Stats will feed directly into N2.')

In [ ]:
# 7. N2 — train generator FROM SCRATCH for 8000 steps with channel-wise whitening
# Single controlled architecture/data-path change vs baseline: latent whitening.
!python -m ray_image.train_generator \
  --manifest data/toy/manifest.jsonl \
  --vae /content/ray_vae_v0_2.pt \
  --whiten-stats /content/n1/vae_latent_stats.json \
  --steps 8000 \
  --batch-size 16 \
  --save /content/ray_image_v0_2_whiten.pt

In [ ]:
# 8. Generate the fixed 12-prompt suite
from pathlib import Path
import subprocess, sys

prompts = [
    ('red_circle', 'a red circle'), ('red_square', 'a red square'), ('red_triangle', 'a red triangle'),
    ('green_circle', 'a green circle'), ('green_square', 'a green square'), ('green_triangle', 'a green triangle'),
    ('blue_circle', 'a blue circle'), ('blue_square', 'a blue square'), ('blue_triangle', 'a blue triangle'),
    ('yellow_circle', 'a yellow circle'), ('yellow_square', 'a yellow square'), ('yellow_triangle', 'a yellow triangle'),
]
out_dir = Path('/content/ray_suite')
out_dir.mkdir(parents=True, exist_ok=True)
for name, prompt in prompts:
    out = out_dir / f'{name}.png'
    cmd = [sys.executable, '-m', 'ray_image.generate', '--checkpoint', '/content/ray_image_v0_2_whiten.pt', '--prompt', prompt, '--steps', '50', '--seed', '42', '--output', str(out)]
    subprocess.run(cmd, check=True)
print(f'Generated {len(list(out_dir.glob("*.png")))} images.')

In [ ]:
# 9. Evaluate the fixed 12-class suite
!python tools/evaluate_toy_suite.py --dir /content/ray_suite

## N3 — Text-conditioning probe (DIAGNOSTIC ONLY, no training)

N1/N2 ruled out VAE latent whitening (endpoint unchanged: color 1.000 / shape
0.333). This probe inspects the **text side** of conditioning on the trained
generator checkpoint to locate where color-vs-shape signal diverges.

It does NOT train and does NOT change any architecture. It reports:
1. Tokenizer distinctness for color/shape words.
2. Token-level text embeddings + padding-mask validity (is the shape token
   visible to cross-attention?).
3. Pooled-embedding distance split (same-shape/diff-color vs
   same-color/diff-shape).
4. DiT conditioning-vector effect for prompts differing ONLY in shape
   (a red circle / a red square / a red triangle).
5. An estimate of whether swapping the shape token materially changes the DiT
   output vs swapping the color token.

Read the printed summary and `/content/n3/text_conditioning_report.json`,
then send the results back to ChatGPT for analysis. **Do not start another
8000-step run yet.**


In [ ]:
# N3 — run the text-conditioning probe (diagnostic; no training)
!mkdir -p /content/n3
!python -m ray_image.probe_text_conditioning \
  --checkpoint /content/ray_image_v0_2_whiten.pt \
  --outdir /content/n3
print('\nReport written to /content/n3/text_conditioning_report.json')


In [ ]:
# 10. Display generated images + compact experiment summary
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display
import json

files = sorted(Path('/content/ray_suite').glob('*.png'))
thumbs = [Image.open(p).convert('RGB').resize((192, 192)) for p in files]
cols = 4
rows = (len(thumbs) + cols - 1) // cols
sheet = Image.new('RGB', (cols * 192, rows * 220), 'white')
draw = ImageDraw.Draw(sheet)
for i, (p, im) in enumerate(zip(files, thumbs)):
    x = (i % cols) * 192
    y = (i // cols) * 220
    sheet.paste(im, (x, y))
    draw.text((x + 5, y + 196), p.stem, fill='black')
display(sheet)

stats = json.loads(Path('/content/n1/vae_latent_stats.json').read_text())
print('\nN1 latent means:', [round(x, 4) for x in stats['per_channel_mean']])
print('N1 latent stds :', [round(x, 4) for x in stats['per_channel_std']])
print('\nArtifacts:')
print('VAE checkpoint : /content/ray_vae_v0_2.pt')
print('N1 stats       : /content/n1/vae_latent_stats.json')
print('N2 checkpoint  : /content/ray_image_v0_2_whiten.pt')
print('Images         : /content/ray_suite')
print('\nN2 RUN COMPLETE — send the final evaluator output + image grid to ChatGPT for audit.')